# Revisi M1 dan M3 — Naskah JIEE Prediksi Stroke
- **M1**: model terbaik untuk Eksperimen 3 dipilih **hanya dari data latih (80%)**, sehingga data uji tidak ikut menentukan model.
- **M3**: analisis sensitivitas **SMOTE-NC** (SMOTE untuk data campuran numerik–kategorikal) dibandingkan SMOTE biasa pada atribut *one-hot*.
- **Statistik**: Wilcoxon + Holm + *rank-biserial*, Friedman + Kendall's W, uji RF terbaik vs regresi logistik.

Prasyarat: folder `hasil_notebook` dari notebook utama sudah ada (dipakai `eksp2_per_fold.csv`). Ubah `DATA_DIR` bila perlu, lalu **Run All**. Waktu: ±30–40 menit (`FAST = True` ±5 menit untuk uji).

## 1. Konfigurasi, data, dan fungsi (sama dengan notebook utama)

In [ ]:
import os, re, glob, json, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

# ==== Struktur repositori: data di ../data, hasil di ../results ====
DATA_DIR = os.path.join("..", "data")
STROKE_CSV = os.path.join(DATA_DIR, "healthcare-dataset-stroke-data.csv")
SCOPUS_GLOB = os.path.join(DATA_DIR, "export_*.csv")       # file ekspor Scopus
OUT = os.path.join("..", "results")
os.makedirs(OUT, exist_ok=True)

SEED = 42
FAST = False          # True = uji cepat (1 pengulangan, 4 kandidat tuning)
USE_OPTUNA = False        # True = tuning dengan Optuna TPE (perlu paket optuna)
USE_EXTRA_MODELS = False  # True = tambah XGBoost & LightGBM (perlu paketnya)
# Default False/False = mereproduksi angka di naskah (RandomizedSearchCV; LR, RF, HGB)
N_REPEATS = 1 if FAST else 2      # repeated stratified 5-fold (outer)
N_ITER    = 4 if FAST else 12     # jumlah kandidat hyperparameter per fold
N_BOOT    = 300 if FAST else 1000

try:
    import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING); HAS_OPTUNA = USE_OPTUNA
except ImportError:
    HAS_OPTUNA = False
print("Optuna aktif:", HAS_OPTUNA, "| model tambahan:", USE_EXTRA_MODELS, "| FAST:", FAST, "| output:", OUT)

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import (StratifiedKFold, RepeatedStratifiedKFold, RandomizedSearchCV,
                                     train_test_split, cross_val_predict, cross_val_score)
from sklearn.metrics import (accuracy_score, roc_auc_score, average_precision_score, matthews_corrcoef, recall_score,
                             precision_score, f1_score, brier_score_loss, roc_curve, confusion_matrix, precision_recall_curve)
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
from scipy.stats import wilcoxon, friedmanchisquare
import matplotlib.pyplot as plt

df = pd.read_csv(STROKE_CSV).drop(columns="id")
df = df[df.gender != "Other"].reset_index(drop=True)
y = df.pop("stroke").values
X = df
NUM = ["age", "avg_glucose_level", "bmi"]; BIN = ["hypertension", "heart_disease"]
CAT = ["gender", "ever_married", "work_type", "Residence_type", "smoking_status"]
print("Data:", X.shape, "| stroke:", y.sum(), f"({y.mean():.2%}) | BMI kosong:", X.bmi.isna().sum())

def make_prep():
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM),
        ("bin", "passthrough", BIN),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), CAT)])

In [ ]:
def smote(X, y, k=5, seed=0):
    # SMOTE (Chawla et al., 2002): interpolasi linear antar tetangga kelas minoritas
    r = np.random.RandomState(seed); Xm = X[y == 1]; n = int((y == 0).sum() - len(Xm))
    if n <= 0: return X, y
    nn = NearestNeighbors(n_neighbors=min(k + 1, len(Xm))).fit(Xm)
    idx = nn.kneighbors(Xm, return_distance=False)[:, 1:]
    i = r.randint(0, len(Xm), n); j = idx[i, r.randint(0, idx.shape[1], n)]; g = r.rand(n, 1)
    return np.vstack([X, Xm[i] + g * (Xm[j] - Xm[i])]), np.r_[y, np.ones(n, int)]

def rus(X, y, seed=0):
    r = np.random.RandomState(seed); pos = np.where(y == 1)[0]; neg = np.where(y == 0)[0]
    keep = np.r_[pos, r.choice(neg, len(pos), replace=False)]
    return X[keep], y[keep]

class Resampled(ClassifierMixin, BaseEstimator):
    # Resampling HANYA saat fit (data latih). Urutan pewarisan (ClassifierMixin dulu) penting
    # agar scorer scikit-learn mengenali ini sebagai classifier.
    def __init__(self, estimator=None, sampler="none", random_state=0):
        self.estimator = estimator; self.sampler = sampler; self.random_state = random_state
    def fit(self, X, y):
        X = np.asarray(X, float); y = np.asarray(y)
        if self.sampler == "smote": X, y = smote(X, y, seed=self.random_state)
        elif self.sampler == "rus": X, y = rus(X, y, seed=self.random_state)
        self.est_ = clone(self.estimator).fit(X, y); self.classes_ = self.est_.classes_; return self
    def predict_proba(self, X): return self.est_.predict_proba(np.asarray(X, float))
    def predict(self, X): return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

SPW = float((y == 0).sum() / (y == 1).sum())   # untuk XGBoost scale_pos_weight

def models(cw=False):
    w = "balanced" if cw else None
    m = {
        "LR": (LogisticRegression(max_iter=3000, class_weight=w), {"C": list(np.logspace(-3, 2, 30))}),
        "RF": (RandomForestClassifier(n_estimators=200, class_weight=w, n_jobs=-1, random_state=SEED),
               {"max_depth": [3, 5, 8, 12, None], "min_samples_leaf": [1, 5, 10, 20, 50], "max_features": ["sqrt", 0.5, 0.8]}),
        "HGB": (HistGradientBoostingClassifier(class_weight=w, random_state=SEED),
                {"learning_rate": list(np.logspace(-2.5, -0.5, 20)), "max_depth": [2, 3, 4, None], "max_leaf_nodes": [7, 15, 31],
                 "min_samples_leaf": [20, 50, 100], "l2_regularization": list(np.logspace(-3, 1, 10))}),
    }
    if not USE_EXTRA_MODELS: return m
    try:
        from xgboost import XGBClassifier
        m["XGB"] = (XGBClassifier(n_estimators=300, eval_metric="logloss", random_state=SEED, n_jobs=-1, scale_pos_weight=(SPW if cw else 1.0)),
                    {"max_depth": [2, 3, 4, 6], "learning_rate": list(np.logspace(-2.5, -0.5, 20)), "subsample": [0.6, 0.8, 1.0],
                     "colsample_bytree": [0.6, 0.8, 1.0], "min_child_weight": [1, 5, 10]})
    except ImportError: pass
    try:
        from lightgbm import LGBMClassifier
        m["LGBM"] = (LGBMClassifier(n_estimators=300, class_weight=w, random_state=SEED, verbose=-1),
                     {"num_leaves": [7, 15, 31], "learning_rate": list(np.logspace(-2.5, -0.5, 20)),
                      "min_child_samples": [10, 20, 50, 100], "reg_lambda": list(np.logspace(-3, 1, 10))})
    except ImportError: pass
    return m

def metrics(yt, p, thr=0.5):
    yh = (p >= thr).astype(int); tn, fp, fn, tp = confusion_matrix(yt, yh, labels=[0, 1]).ravel()
    return dict(Accuracy=accuracy_score(yt, yh), ROC_AUC=roc_auc_score(yt, p), PR_AUC=average_precision_score(yt, p),
                Recall=recall_score(yt, yh, zero_division=0), Specificity=tn / (tn + fp), Precision=precision_score(yt, yh, zero_division=0),
                F1=f1_score(yt, yh, zero_division=0), MCC=matthews_corrcoef(yt, yh), Brier=brier_score_loss(yt, p))

def boot_ci(yt, p, fn, B=N_BOOT, seed=SEED):
    r = np.random.RandomState(seed); v = []
    for _ in range(B):
        i = r.randint(0, len(yt), len(yt))
        if yt[i].sum() > 0: v.append(fn(yt[i], p[i]))
    return np.percentile(v, [2.5, 97.5])

print("Model tersedia:", list(models().keys()))

In [ ]:
def tune(pipe, space, Xtr, ytr, n_iter=N_ITER, seed=SEED):
    # Kembalikan pipeline terbaik yang sudah di-fit pada Xtr (pemilihan parameter hanya memakai Xtr)
    sp = {f"clf__estimator__{k}": v for k, v in space.items()}
    if HAS_OPTUNA:
        def objective(trial):
            params = {}
            for k, v in sp.items():
                choice = trial.suggest_categorical(k, list(range(len(v))))   # indeks agar aman untuk None/float
                params[k] = v[choice]
            return cross_val_score(clone(pipe).set_params(**params), Xtr, ytr, cv=inner, scoring="average_precision", error_score="raise").mean()
        study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed))
        study.optimize(objective, n_trials=n_iter, show_progress_bar=False)
        best = {k: sp[k][study.best_params[k]] for k in sp}
        return clone(pipe).set_params(**best).fit(Xtr, ytr), best
    rs = RandomizedSearchCV(pipe, sp, n_iter=n_iter, scoring="average_precision", cv=inner,
                            random_state=seed, n_jobs=1, error_score="raise").fit(Xtr, ytr)
    return rs.best_estimator_, rs.best_params_

# uji cepat: scorer harus bekerja (mencegah tuning diam-diam gagal)
_p = Pipeline([("prep", make_prep()), ("clf", Resampled(models(True)["LR"][0], "none"))]).fit(X, y)
from sklearn.metrics import get_scorer
assert abs(get_scorer("average_precision")(_p, X, y) - average_precision_score(y, _p.predict_proba(X)[:, 1])) < 1e-9
print("Scorer OK")

## 2. SMOTE-NC dan praproses dengan kategori tetap
Kategori *one-hot* dikunci dari daftar label (bukan dari nilai data latih) agar posisi kolom sama di setiap *fold*. SMOTE-NC menginterpolasi atribut numerik dan mengambil **nilai terbanyak di antara k tetangga** untuk atribut kategorikal, dengan jarak yang memberi penalti median simpangan baku numerik untuk setiap atribut kategorikal yang berbeda (Chawla dkk., 2002).

In [ ]:
CATS = [sorted(X[c].unique()) for c in CAT]
def make_prep():
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM),
        ("bin", "passthrough", BIN),
        ("cat", OneHotEncoder(categories=CATS, drop="first", handle_unknown="ignore", sparse_output=False), CAT)])

NUM_IDX = list(range(len(NUM)))
GROUPS = [[len(NUM)], [len(NUM) + 1]]          # hipertensi, penyakit jantung (biner)
_start = len(NUM) + 2
for cats in CATS:
    GROUPS.append(list(range(_start, _start + len(cats) - 1))); _start += len(cats) - 1
print("Jumlah fitur setelah praproses:", _start, "| kelompok kategorikal:", GROUPS)

def smotenc(X, y, k=5, seed=0):
    r = np.random.RandomState(seed); Xm = X[y == 1]; n = int((y == 0).sum() - len(Xm))
    if n <= 0: return X, y
    med_sd = np.median(Xm[:, NUM_IDX].std(axis=0))
    dn = ((Xm[:, None, NUM_IDX] - Xm[None, :, NUM_IDX]) ** 2).sum(-1)
    dc = sum((np.abs(Xm[:, None, g] - Xm[None, :, g]).sum(-1) > 0).astype(float) for g in GROUPS)
    D = np.sqrt(dn + dc * med_sd ** 2); np.fill_diagonal(D, np.inf)
    nn = np.argsort(D, axis=1)[:, :k]
    i = r.randint(0, len(Xm), n); j = nn[i, r.randint(0, k, n)]; gap = r.rand(n, 1)
    S = Xm[i].copy()
    S[:, NUM_IDX] = Xm[i][:, NUM_IDX] + gap * (Xm[j][:, NUM_IDX] - Xm[i][:, NUM_IDX])
    for g in GROUPS:                              # nilai terbanyak di antara k tetangga
        neigh = Xm[:, g][nn[i]]                   # (n, k, |g|)
        for t in range(n):
            vals, cnt = np.unique(neigh[t], axis=0, return_counts=True)
            S[t, g] = vals[cnt.argmax()]
    return np.vstack([X, S]), np.r_[y, np.ones(n, int)]

class Resampled2(Resampled):
    def fit(self, X, y):
        if self.sampler != "smotenc": return super().fit(X, y)
        X, y = smotenc(np.asarray(X, float), np.asarray(y), seed=self.random_state)
        self.est_ = clone(self.estimator).fit(X, y); self.classes_ = self.est_.classes_; return self

# cek: atribut kategorikal sintetis harus 0/1 (tidak pecahan)
_Xp = make_prep().fit_transform(X); _Xs, _ys = smotenc(_Xp, y, seed=SEED)
_cat_cols = [c for g in GROUPS for c in g]
print("Nilai unik atribut kategorikal sintetis:", np.unique(_Xs[len(y):, _cat_cols]))
_Xs2, _ = smote(_Xp, y, seed=SEED)
print("Pembanding SMOTE biasa — proporsi nilai pecahan:", np.mean((_Xs2[len(y):, _cat_cols] % 1) != 0).round(3))

## 3. M3 — SMOTE-NC vs SMOTE (nested CV, fold luar sama dengan notebook utama)

In [ ]:
t0 = time.time()
outer = RepeatedStratifiedKFold(n_splits=5, n_repeats=N_REPEATS, random_state=SEED)
inner = StratifiedKFold(3, shuffle=True, random_state=SEED)
rows4, folds4 = [], []
for name, (est, space) in models(False).items():
    for mode in ["default", "tuned"]:
        P = np.zeros(len(y)); cnt = np.zeros(len(y))
        for f, (tr, te) in enumerate(outer.split(X, y)):
            pipe = Pipeline([("prep", make_prep()), ("clf", Resampled2(est, "smotenc", SEED))])
            if mode == "tuned": pipe, _ = tune(pipe, space, X.iloc[tr], y[tr], seed=SEED + f)
            else: pipe.fit(X.iloc[tr], y[tr])
            p = pipe.predict_proba(X.iloc[te])[:, 1]; P[te] += p; cnt[te] += 1
            folds4.append(dict(Strategi="smotenc", Model=name, Mode=mode, Fold=f, PR_AUC=average_precision_score(y[te], p),
                               ROC_AUC=roc_auc_score(y[te], p), MCC=matthews_corrcoef(y[te], (p >= .5).astype(int))))
        P /= cnt; m = metrics(y, P); lo, hi = boot_ci(y, P, average_precision_score); lo2, hi2 = boot_ci(y, P, roc_auc_score)
        rows4.append(dict(Strategi="smotenc", Model=name, Mode=mode, **m, PR_AUC_CI=f"{lo:.3f}-{hi:.3f}", ROC_AUC_CI=f"{lo2:.3f}-{hi2:.3f}"))
        print(f"[{time.time()-t0:5.0f}s] smotenc {name:4s} {mode:7s} ROC={m['ROC_AUC']:.3f} PR={m['PR_AUC']:.3f} MCC={m['MCC']:.3f} Brier={m['Brier']:.3f}")
r4 = pd.DataFrame(rows4); r4.to_csv(f"{OUT}/eksp4_smotenc.csv", index=False)
f4 = pd.DataFrame(folds4); f4.to_csv(f"{OUT}/eksp4_smotenc_per_fold.csv", index=False)
display(r4.round(3))

## 4. Statistik lengkap (dipakai di naskah)
Wilcoxon (statistik T) + koreksi Holm per model + korelasi *rank-biserial*; Friedman + Kendall's W (kini 5 strategi termasuk SMOTE-NC); SMOTE vs SMOTE-NC; RF terbaik vs regresi logistik.

In [ ]:
from scipy.stats import rankdata
def rrb(d):
    d = np.asarray(d); d = d[d != 0]; r = rankdata(np.abs(d)); return (r[d > 0].sum() - r[d < 0].sum()) / r.sum()
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
    for k, i in enumerate(o): run = max(run, (m - k) * p[i]); adj[i] = min(1, run)
    return adj
fr = pd.read_csv(f"{OUT}/eksp2_per_fold.csv")
fr_all = pd.concat([fr, pd.read_csv(f"{OUT}/eksp4_smotenc_per_fold.csv")], ignore_index=True)
get = lambda s, m, mode: fr_all[(fr_all.Strategi == s) & (fr_all.Model == m) & (fr_all.Mode == mode)].sort_values("Fold").PR_AUC.values

# (a) default vs tuned, termasuk SMOTE-NC
rows = []
for (s, m), g in fr_all.groupby(["Strategi", "Model"]):
    a, b = get(s, m, "default"), get(s, m, "tuned"); w = wilcoxon(b, a)
    rows.append(dict(Model=m, Strategi=s, mean_delta=(b - a).mean(), T=w.statistic, p=w.pvalue, r_rb=rrb(b - a), unggul=int((b > a).sum())))
wt = pd.DataFrame(rows); wt["p_holm_model"] = wt.groupby("Model").p.transform(lambda s: holm(s.values)); wt["p_holm_semua"] = holm(wt.p.values)
wt.to_csv(f"{OUT}/stat_wilcoxon_tuning.csv", index=False); display(wt.round(4))

# (b) Friedman 5 strategi (tuned) + Kendall's W
rows = []
for m, g in fr_all[fr_all.Mode == "tuned"].groupby("Model"):
    piv = g.pivot(index="Fold", columns="Strategi", values="PR_AUC"); f = friedmanchisquare(*[piv[c] for c in piv.columns])
    n, k = piv.shape
    rows.append(dict(Model=m, chi2=f.statistic, df=k - 1, p=f.pvalue, Kendall_W=f.statistic / (n * (k - 1)),
                     **{f"rank_{c}": v for c, v in piv.rank(axis=1, ascending=False).mean().items()}))
ft = pd.DataFrame(rows); ft.to_csv(f"{OUT}/stat_friedman.csv", index=False); display(ft.round(4))

# (c) M3: SMOTE vs SMOTE-NC (tuned), per model
rows = []
for m in ["LR", "RF", "HGB"]:
    a, b = get("smote", m, "tuned"), get("smotenc", m, "tuned"); w = wilcoxon(b, a)
    rows.append(dict(Model=m, PR_AUC_smote=a.mean(), PR_AUC_smotenc=b.mean(), mean_delta=(b - a).mean(), T=w.statistic, p=w.pvalue, r_rb=rrb(b - a)))
sn = pd.DataFrame(rows); sn["p_holm"] = holm(sn.p.values); sn.to_csv(f"{OUT}/stat_smote_vs_smotenc.csv", index=False); display(sn.round(4))

# (d) RF class weight tuned vs regresi logistik
best = get("class_weight", "RF", "tuned"); rows = []
for lab, (s, m, mode) in {"LR tanpa penanganan (bawaan)": ("none", "LR", "default"), "LR tanpa penanganan (tuned)": ("none", "LR", "tuned"),
                          "LR class weight (bawaan)": ("class_weight", "LR", "default")}.items():
    o = get(s, m, mode); w = wilcoxon(best, o)
    rows.append(dict(Pembanding=lab, mean_delta=(best - o).mean(), T=w.statistic, p=w.pvalue, r_rb=rrb(best - o), RF_unggul=int((best > o).sum())))
bv = pd.DataFrame(rows); bv.to_csv(f"{OUT}/stat_rf_vs_lr.csv", index=False); display(bv.round(4))

## 5. M1 — Pemilihan model hanya dari data latih, lalu evaluasi pada data uji
Data dibagi 80:20 **lebih dulu**. Seluruh kombinasi strategi × model dievaluasi dengan *nested CV* di dalam 80% data latih. Model terbaik (PR-AUC tertinggi) dilatih ulang pada data latih, dikalibrasi, dan baru diuji pada data uji.

In [ ]:
t0 = time.time()
xtr, xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
outer_tr = RepeatedStratifiedKFold(n_splits=5, n_repeats=N_REPEATS, random_state=SEED)
sel = []
for strat in ["none", "smote", "rus", "class_weight", "smotenc"]:
    samp = strat if strat in ("smote", "rus", "smotenc") else "none"
    for name, (est, space) in models(cw=(strat == "class_weight")).items():
        P = np.zeros(len(ytr)); cnt = np.zeros(len(ytr))
        for f, (tr, te) in enumerate(outer_tr.split(xtr, ytr)):
            pipe, _ = tune(Pipeline([("prep", make_prep()), ("clf", Resampled2(est, samp, SEED))]), space, xtr.iloc[tr], ytr[tr], seed=SEED + f)
            P[te] += pipe.predict_proba(xtr.iloc[te])[:, 1]; cnt[te] += 1
        P /= cnt; sel.append(dict(Strategi=strat, Model=name, PR_AUC_latih=average_precision_score(ytr, P), ROC_AUC_latih=roc_auc_score(ytr, P)))
        print(f"[{time.time()-t0:5.0f}s] {strat:12s} {name:4s} PR-AUC (nested CV, data latih) = {sel[-1]['PR_AUC_latih']:.3f}")
sel = pd.DataFrame(sel).sort_values("PR_AUC_latih", ascending=False); sel.to_csv(f"{OUT}/eksp3b_seleksi_data_latih.csv", index=False)
best_s, best_m = sel.iloc[0].Strategi, sel.iloc[0].Model; print("Model terpilih (hanya dari data latih):", best_s, best_m)
display(sel.round(3))

In [ ]:
def wilson(k, n, z=1.959964):
    p = k / n; den = 1 + z * z / n; c = (p + z * z / (2 * n)) / den; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return c - h, c + h
est, space = models(cw=(best_s == "class_weight"))[best_m]; samp = best_s if best_s in ("smote", "rus", "smotenc") else "none"
raw, best_params = tune(Pipeline([("prep", make_prep()), ("clf", Resampled2(est, samp, SEED))]), space, xtr, ytr, n_iter=N_ITER * 2)
print("Parameter terbaik:", best_params)
cv5 = StratifiedKFold(5, shuffle=True, random_state=SEED)
oof_raw = cross_val_predict(clone(raw), xtr, ytr, cv=cv5, method="predict_proba")[:, 1]
iso = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(oof_raw, ytr)
p_raw = raw.predict_proba(xte)[:, 1]; p_cal = iso.predict(p_raw)
rows3, rng = [], np.random.RandomState(SEED)
for lab, oof, p in [("Tanpa kalibrasi", oof_raw, p_raw), ("Kalibrasi isotonik", iso.predict(oof_raw), p_cal)]:
    fpr, tpr, th = roc_curve(ytr, oof); thr = float(th[np.argmax(tpr - fpr)])
    for tname, t in [("0.5", 0.5), (f"Youden OOF latih = {thr:.3f}", thr)]:
        m = metrics(yte, p, t); yh = (p >= t).astype(int)
        tp = int(((yh == 1) & (yte == 1)).sum()); fn = int(((yh == 0) & (yte == 1)).sum()); tn = int(((yh == 0) & (yte == 0)).sum()); fp = int(((yh == 1) & (yte == 0)).sum())
        ci = {}
        for key, (k_, n_) in {"Recall": (tp, tp + fn), "Specificity": (tn, tn + fp), "Precision": (tp, tp + fp)}.items():
            ci[key + "_CI"] = "-".join(f"{v:.3f}" for v in wilson(k_, n_)) if n_ > 0 else ""
        bs = {"ROC_AUC": [], "PR_AUC": [], "MCC": [], "Brier": []}
        for _ in range(2000):
            i = rng.randint(0, len(yte), len(yte))
            if yte[i].sum() == 0: continue
            bs["ROC_AUC"].append(roc_auc_score(yte[i], p[i])); bs["PR_AUC"].append(average_precision_score(yte[i], p[i]))
            bs["MCC"].append(matthews_corrcoef(yte[i], yh[i])); bs["Brier"].append(brier_score_loss(yte[i], p[i]))
        for key, v in bs.items(): ci[key + "_CI"] = "-".join(f"{x:.3f}" for x in np.percentile(v, [2.5, 97.5]))
        rows3.append(dict(Model=f"{best_m}+{best_s}", Versi=lab, Ambang=tname, TP=tp, FN=fn, TN=tn, FP=fp, **m, **ci))
r3b = pd.DataFrame(rows3); r3b.to_csv(f"{OUT}/eksp3b_kalibrasi_ambang.csv", index=False)
json.dump({k: str(v) for k, v in best_params.items()}, open(f"{OUT}/eksp3b_best_params.json", "w"), indent=1)
print("Probabilitas terkalibrasi maksimum pada data uji:", p_cal.max().round(3)); display(r3b.round(3).T)

In [ ]:
pi = permutation_importance(raw, xte, yte, scoring="average_precision", n_repeats=20, random_state=SEED)
imp = pd.DataFrame(dict(Fitur=X.columns, Importance_mean=pi.importances_mean, Importance_sd=pi.importances_std)).sort_values("Importance_mean", ascending=False)
imp.to_csv(f"{OUT}/eksp3b_permutation_importance.csv", index=False); display(imp.round(4))
fc = lambda v: f"{v:.3f}".replace(".", ",")
fig, ax = plt.subplots(figsize=(5.2, 5))
for lab, p, col in [("Tanpa kalibrasi", p_raw, "#8c8c8c"), ("Kalibrasi isotonik", p_cal, "#1f5fa8")]:
    a, b = calibration_curve(yte, p, n_bins=8, strategy="quantile"); ax.plot(b, a, "o-", color=col, label=f"{lab} (Brier {fc(brier_score_loss(yte, p))})")
ax.plot([0, 1], [0, 1], "k--", lw=.8, label="Kalibrasi sempurna"); ax.set_xlabel("Probabilitas stroke hasil prediksi"); ax.set_ylabel("Proporsi stroke teramati")
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.legend(loc="upper left", fontsize=8.5); fig.tight_layout(); fig.savefig(f"{OUT}/gambar_kalibrasi_M1.png", dpi=300); plt.show()
print("Selesai. File baru di:", OUT)
for f_ in ["eksp4_smotenc.csv", "eksp4_smotenc_per_fold.csv", "stat_wilcoxon_tuning.csv", "stat_friedman.csv", "stat_smote_vs_smotenc.csv",
           "stat_rf_vs_lr.csv", "eksp3b_seleksi_data_latih.csv", "eksp3b_kalibrasi_ambang.csv", "eksp3b_best_params.json", "eksp3b_permutation_importance.csv", "gambar_kalibrasi_M1.png"]:
    print("  -", f_)